# Change Detection & Temporal Analysis
**PyGeoVision v2.0 — Notebook 04**

## Real-World Problem
Disaster response teams need to identify damaged areas after Hurricane
by comparing satellite imagery taken before and after the event.

## Learning Objectives
1. Acquire bi-temporal Sentinel-2 imagery
2. Run ChangeFormer (siamese transformer) for pixel-accurate change maps
3. Use DINOv3 embeddings for semantic change understanding
4. Apply Prithvi multi-temporal change detection
5. Threshold tuning and false-alarm reduction
6. Export change masks and statistics reports

```bash
pip install "pygeovision[geo,train,foundation]"
```

In [ ]:
import pygeovision as pgv
import numpy as np, rasterio, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/04_change_detection")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Haiti — bi-temporal change study
BBOX    = (-72.50, 18.30, -72.00, 18.70)
CITY    = "Port-au-Prince, Haiti"
PRE_DT  = ("2023-01-01", "2023-06-30")
POST_DT = ("2024-01-01", "2024-06-30")

client = pgv.PyGeoVision()
print(client)
print(f"\nStudy area : {CITY}")

## Step 1: Acquire Bi-Temporal Imagery

In [ ]:
def get_best_scene(date_range, label):
    results = client.search(
        bbox=BBOX, date_range=date_range,
        providers=["planetary_computer"], cloud_cover_max=10,
        sort_by="cloud_cover", limit=3,
    )
    print(f"  {label}: {len(results)} scenes found")
    if results:
        dl = client.download(
            results[:1], output_dir=BASE_DIR / label.lower().replace(" ","_"),
            post_process=["reproject:EPSG:32618","cog"],
        )
        if dl and dl[0].success:
            print(f"    Downloaded: {Path(dl[0].path).name}")
            return dl[0].path
    return None

pre_scene  = get_best_scene(PRE_DT,  "Pre-event 2023")
post_scene = get_best_scene(POST_DT, "Post-event 2024")
print(f"\nPre-event  : {pre_scene or 'not available'}")
print(f"Post-event : {post_scene or 'not available'}")

## Step 2: ChangeFormer Detection

In [ ]:
from pygeovision.models.change_detection.changeformer import ChangeFormer

# ChangeFormer: siamese transformer architecture
# Input: two temporally separated images
# Output: binary change mask + change statistics
print("ChangeFormer specifications:")
print("  Architecture: Siamese MiT (Mix Transformer) encoder")
print("  Variants:     MIT-B0 (3.7M params) to MIT-B4 (84.7M params)")
print("  Input:        2x image patches, 4 channels each")
print("  Output:       Binary change mask")
print()

cd = ChangeFormer(num_classes=2, in_channels=4, device="cpu")
print(f"Loaded: {type(cd).__name__}")

if pre_scene and post_scene and Path(pre_scene).exists() and Path(post_scene).exists():
    print("\nRunning ChangeFormer on real scenes...")
    result = cd.detect(pre_scene, post_scene,
                        output_path=str(BASE_DIR/"change_map_cf.tif"))
    change_pct = result.get("change_pct", 0)
    print(f"  Changed area: {change_pct:.2f}%")
    CHANGE_AVAILABLE = True
else:
    # Generate synthetic change map
    print("\nGenerating synthetic demo change map...")
    H, W = 256, 256
    np.random.seed(42)
    change_map = np.zeros((H,W), dtype=np.uint8)
    rng = np.random.default_rng(42)
    for _ in range(25):
        r,c  = rng.integers(10, H-40, size=2)
        rh,rw= rng.integers(6, 30, size=2)
        change_map[r:r+rh, c:c+rw] = 1
    change_pct = float(change_map.mean() * 100)
    import rasterio.transform as rt
    cp = BASE_DIR/"change_map_cf.tif"
    with rasterio.open(cp,'w',driver='GTiff',height=H,width=W,count=1,
                       dtype='uint8', crs='EPSG:32618',
                       transform=rt.from_bounds(*BBOX,W,H)) as dst:
        dst.write(change_map[np.newaxis])
    CHANGE_AVAILABLE = True
    print(f"  Demo change area: {change_pct:.2f}%")

## Step 3: DINOv3 Semantic Change Detection

In [ ]:
from pygeovision.ai.geoai.dinov3_proxy import DINOv3Proxy
from pygeovision.models.foundation.dinov3 import DINOv3Backbone

print("DINOv3 Semantic Change Detection:")
print()
print("Traditional change detection:")
print("  Compares pixel VALUES -> sensitive to illumination, sensor noise")
print()
print("DINOv3 embedding-based change detection:")
print("  Compares SEMANTIC FEATURES -> understands WHAT changed")
print("  Distinguishes: seasonal vegetation vs deforestation")
print("                 shadow artifacts vs real building collapse")
print("                 cloud contamination vs actual change")
print()
print("Pipeline:")
print("  1. Extract DINOv3 patch features for both images")
print("  2. Compute cosine distance per patch pair")
print("  3. Threshold distance map -> semantic change mask")
print()

# Show API
proxy = DINOv3Proxy()
print("API usage:")
print("  proxy = DINOv3Proxy()")
print("  proxy.load('dinov3_vitl16_sat')")
print()
print("  # Embedding extraction")
print("  emb_pre  = proxy.extract_embeddings(pre_scene)   # (1, 1024)")
print("  emb_post = proxy.extract_embeddings(post_scene)  # (1, 1024)")
print()
print("  # Semantic similarity")
print("  from sklearn.metrics.pairwise import cosine_similarity")
print("  sim = cosine_similarity(emb_pre, emb_post)[0][0]")
print("  print(f'Scene similarity: {sim:.3f}')")
print("  print('High sim -> little change, Low sim -> major change')")

## Step 4: Prithvi Multi-Temporal Change

In [ ]:
from pygeovision.models.foundation.prithvi import PrithviMultiTemporal

mt = PrithviMultiTemporal("prithvi_eo_2_0")
print("Prithvi-EO-2.0 Multi-Temporal Change Detection:")
print()
print("Approach: temporal attention over multiple frames simultaneously")
print("Advantage: handles seasonal variability better than bi-temporal models")
print()

if pre_scene and post_scene and Path(pre_scene).exists() and Path(post_scene).exists():
    mt_result = mt.detect_change(pre_scene, post_scene,
                                   output_path=str(BASE_DIR/"change_map_prithvi.tif"))
    print(f"Prithvi change: {mt_result.get('change_pct',0):.1f}%")
else:
    print("Production usage:")
    print("  result = mt.detect_change(pre_scene, post_scene)")
    print("  print(f'Changed: {result["change_pct"]:.1f}%')")
    print()
    print("  # For multi-date (>2 images):")
    print("  result = mt.process_time_series(images, dates=dates)")

## Step 5: Threshold Analysis

In [ ]:
# Load change probability map and analyse threshold effects
CHANGE_PATH = BASE_DIR / "change_map_cf.tif"

if CHANGE_PATH.exists():
    with rasterio.open(CHANGE_PATH) as src:
        change_map = src.read(1)
else:
    # Demo
    np.random.seed(42)
    change_map = np.random.rand(256, 256)

print("Threshold sensitivity analysis:")
print(f"{'Threshold':>10} {'Changed%':>10} {'Changed km2':>12} {'Recall':>8}")
print("─" * 45)

# If binary map (0/1), convert to probability for threshold analysis
if change_map.max() <= 1 and change_map.dtype == np.uint8:
    prob_map = change_map.astype(float)
else:
    prob_map = (change_map - change_map.min()) / (change_map.max() - change_map.min() + 1e-8)

# Compute pixel area (approximate for WGS84 bbox)
lon_span = abs(BBOX[2] - BBOX[0])
lat_span = abs(BBOX[3] - BBOX[1])
km2_per_pixel = (lon_span * 111.32) * (lat_span * 111.32) / (prob_map.size)

for thr in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    binary   = (prob_map >= thr)
    pct      = binary.mean() * 100
    km2      = binary.sum() * km2_per_pixel
    recall   = pct / max(change_pct, 0.01)   # Relative recall
    flag     = " <-- recommended" if abs(thr - 0.50) < 0.01 else ""
    print(f"  {thr:>9.2f}  {pct:>9.1f}%  {km2:>11.2f}  {recall:>7.1%}{flag}")

## Step 6: Visualisation

In [ ]:
CHANGE_PATH = BASE_DIR / "change_map_cf.tif"
if CHANGE_PATH.exists():
    with rasterio.open(CHANGE_PATH) as src:
        change_map = src.read(1)
else:
    np.random.seed(0)
    change_map = np.random.randint(0,2,(256,256), dtype=np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Pre (simulated)
np.random.seed(1)
fake_pre = np.clip(np.random.randn(128,128,3)*0.12 + 0.38, 0, 1)
axes[0].imshow(fake_pre)
axes[0].set_title("Pre-Event (2023)\nSentinel-2 True Colour", fontsize=11, fontweight='bold')
axes[0].axis('off')

# Post (simulated damage)
fake_post = fake_pre.copy()
np.random.seed(2)
dm = change_map if change_map.shape == (128,128) else change_map[:128,:128]
if dm.shape != (128,128):
    dm_r = np.zeros((128,128),dtype=np.uint8)
    h = min(128, dm.shape[0]); w = min(128, dm.shape[1])
    dm_r[:h,:w] = dm[:h,:w]
    dm = dm_r
fake_post[dm > 0] = [0.85, 0.78, 0.65]
axes[1].imshow(fake_post)
axes[1].set_title("Post-Event (2024)\nSentinel-2 True Colour", fontsize=11, fontweight='bold')
axes[1].axis('off')

# Change map
from_map = change_map if change_map.shape[0] >= 128 else change_map
axes[2].imshow(from_map, cmap=mcolors.ListedColormap(['#ECF0F1','#E74C3C']),
                vmin=0, vmax=1)
pct_val = float(from_map.mean() * 100) if from_map.dtype == np.uint8 else change_pct
axes[2].set_title(f"Change Detection Map\n({pct_val:.1f}% changed area)",
                   fontsize=11, fontweight='bold')
axes[2].axis('off')

import matplotlib.patches as mpatches
legend = [mpatches.Patch(color='#ECF0F1', label='No change'),
          mpatches.Patch(color='#E74C3C', label='Change detected')]
axes[2].legend(handles=legend, loc='lower right', fontsize=9)

plt.suptitle(f"ChangeFormer Bi-Temporal Analysis — {CITY}", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR/"change_detection_result.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {BASE_DIR/'change_detection_result.png'}")

## Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 04 — Change Detection & Temporal Analysis")
print("=" * 60)
print()
print("Methods compared:")
print("  ChangeFormer   : Best for structural/building change")
print("  ChangeSTAR     : Change + semantic segmentation combined")
print("  DINOv3 diff    : Semantic change (illumination-robust)")
print("  Prithvi MT     : Multi-temporal, seasonal-aware")
print()
print(f"Change detected: {change_pct:.1f}% of study area")
print()
print("Next: 05_agricultural_crop_monitoring.ipynb")